In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iammustafatz/diabetes-prediction-dataset")

print("Path to dataset files:", path)

c:\Users\minh tai\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\minh tai\.cache\kagglehub\datasets\iammustafatz\diabetes-prediction-dataset\versions\1


In [4]:
data_path = path + '/diabetes_prediction_dataset.csv'

In [ ]:
from EDA import eda_process
from preprocessing import preprocess
import pandas as pd

df1, df2 = eda_process(data_path, target_col='diabetes')

processed_results = preprocess(df1, df2, target_column='diabetes', random_state=42)

X_train = processed_results['Dataset_2']['X_train']
y_train = processed_results['Dataset_2']['y_train']
X_val = processed_results['Dataset_2']['X_val']
y_val = processed_results['Dataset_2']['y_val']
X_test = processed_results['Dataset_2']['X_test']
y_test = processed_results['Dataset_2']['y_test']

print("Kích thước tập Train (đã qua SMOTE):", X_train.shape)
print("Kích thước tập Test:", X_test.shape)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

svc_base = LinearSVC(dual=False, random_state=42, max_iter=2000)

param_distributions = {
    'C': np.logspace(-4, 4, 20) 
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=svc_base,
    param_distributions=param_distributions,
    n_iter=10, #
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_val, y_val)

best_C = random_search.best_params_['C']
tuned_lambda = 1 / best_C  

print(f"- C tốt nhất (Library): {best_C:.6f}")
print(f"- Lambda tham khảo cho Manual: {tuned_lambda:.6f}")
print(f"- Điểm F1 ứng với C này: {random_search.best_score_:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import f1_score, make_scorer, classification_report
from scipy.stats import loguniform
import time
import os

def run_ultimate_search():
    data_dir = r"f:\X-FILE\Code_UNI\Python\Machine Learning\CuoiKy\Tai\processed_data"
    X_path = os.path.join(data_dir, "X_train_Dataset_2.csv")
    y_path = os.path.join(data_dir, "y_train_Dataset_2.csv")
    X_test_path = os.path.join(data_dir, "X_test_Dataset_2.csv")
    y_test_path = os.path.join(data_dir, "y_test_Dataset_2.csv")
    
    X = pd.read_csv(X_path)
    y = pd.read_csv(y_path).iloc[:, 0]
    X_test = pd.read_csv(X_test_path)
    y_test = pd.read_csv(y_test_path).iloc[:, 0]
    
    X_sample, _, y_sample, _ = train_test_split(
        X, y, train_size=0.20, stratify=y, random_state=42
    )
    print(f"Sample size: {X_sample.shape[0]} samples")
    
    param_dist = {
        'C': loguniform(1000, 10000),
        'gamma': loguniform(0.001, 0.1),
        'kernel': ['rbf']
    }
    
    svc = SVC(probability=False, random_state=42)
    f1_scorer = make_scorer(f1_score)
    
    random_search = RandomizedSearchCV(
        svc, 
        param_distributions=param_dist, 
        n_iter=20, 
        cv=3, 
        scoring=f1_scorer, 
        verbose=1, 
        n_jobs=-1,
        random_state=42
    )
    
    start_time = time.time()
    print("Searching for the absolute best params (20 iterations)...")
    random_search.fit(X_sample, y_sample)
    end_time = time.time()
    
    print("\n" + "="*50)
    print("ULTIMATE TUNING COMPLETED")
    print("="*50)
    best_params = random_search.best_params_
    print(f"Absolute Best Parameters found: {best_params}")
    print(f"Best Validation F1-Score: {random_search.best_score_:.4f}")
    
    print("\nEvaluating on Test Set...")
    best_svc = random_search.best_estimator_
    y_pred = best_svc.predict(X_test)
    test_f1 = f1_score(y_test, y_pred)
    test_report = classification_report(y_test, y_pred, digits=4)
    
    print("\nFinal Performance on Test Set:")
    print(test_report)
    print(f"Total Tuning Time: {end_time - start_time:.2f} seconds")
    print("="*50)
    
    result_file = r"f:\X-FILE\Code_UNI\Python\Machine Learning\CuoiKy\Tai\docs\ablation\ultimate_svc_params.txt"
    with open(result_file, "w", encoding='utf-8') as f:
        f.write("--- ULTIMATE SVC SEARCH RESULTS ---\n")
        f.write(f"Best Parameters: {best_params}\n")
        f.write(f"Validation F1: {random_search.best_score_:.4f}\n")
        f.write(f"Test F1: {test_f1:.4f}\n")
        f.write("\nClassification Report:\n")
        f.write(test_report)
        f.write(f"\nTuning Time: {end_time - start_time:.22f}s\n")

if __name__ == "__main__":
    run_ultimate_search()

8. Kết quả đánh giá cuối cùng (Ultimate Evaluation)
Dựa trên kết quả Tuning tối ưu:
- **C**: 9239.1503
- **Gamma**: 0.04138
- **Kernel**: RBF
- **F1-Score (Class 1)**: ~0.665
- **Accuracy**: ~0.929

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

C_opt = 9239.150319627252
gamma_opt = 0.041380401125610165
kernel_opt = 'rbf'

print("=" * 80)
print("TIẾN HÀNH ĐÁNH GIÁ CHI TIẾT MÔ HÌNH SVM TỐI ƯU")
print(f"Tham số: C={C_opt}, gamma={gamma_opt}, kernel={kernel_opt}")
print("=" * 80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sample_size = 25000
indices = np.random.choice(X_train_scaled.shape[0], sample_size, replace=False)
X_train_final = X_train_scaled[indices]
y_train_final = y_train.iloc[indices].values

svc_final = SVC(C=C_opt, gamma=gamma_opt, kernel=kernel_opt, probability=True, random_state=42)
svc_final.fit(X_train_final, y_train_final)

print("Đang thực hiện dự đoán trên tập Test...")
y_pred = svc_final.predict(X_test_scaled)
y_proba = svc_final.predict_proba(X_test_scaled)[:, 1]
y_true = y_test.values

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, digits=4))

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle('PHÂN TÍCH HIỆU SUẤT MÔ HÌNH SVM (ULTIMATE PARAMS)', fontsize=18, fontweight='bold')

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], annot_kws={"size": 14})
axes[0].set_title('Ma trận nhầm lẫn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

fpr, tpr, _ = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_title('Đường cong ROC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc="lower right")

precision, recall, _ = precision_recall_curve(y_true, y_proba)
pr_auc = average_precision_score(y_true, y_proba)
axes[2].plot(recall, precision, color='purple', lw=3, label=f'PRC (AP = {pr_auc:.4f})')
axes[2].set_title('Đường cong Precision-Recall', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()
